### Environment Setup

This notebook was tested with:
- Python 3.10
- Tensorflow 2.8.0

All required Python packages are listed in `requirements.txt`.

In [ ]:
# Uncomment if running in a fresh environment (e.g. Google Colab)
# !pip install -r requirements.txt

In [2]:
# Import required modules and libraries
import os
import sys
import random
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import precision_score, recall_score, f1_score, jaccard_score, matthews_corrcoef
from keras.callbacks import ModelCheckpoint as checkpoint
from tensorflow.keras.optimizers import Adam
import tensorflow as tf
from tensorflow.keras import backend as K
os.environ['SM_FRAMEWORK'] = 'tf.keras'
import segmentation_models as sm

sys.path.append("D:\\M.Tech\\Landslide_DL\\Phase3_GitHub")
from Model_Architecture import unet_triple as model_1, unetpp_triple as model_2, unetpp_triple_eca as model_3, unetpp_triple_deepsup_eca as model_4, unetpp_triple_deepsup_eca_aspp as model_5
from Functions import MCC, tversky_loss

seed = 42
tf.random.set_seed(seed)
np.random.seed(seed)
random.seed(seed)

In [ ]:
# Load the datasets and print their shape
mss = np.load("D:\\M.Tech\\Landslide_DL\\Phase3_GitHub\\Dataset\\MSS_SAMPLE.npy")
dem = np.load("D:\\M.Tech\\Landslide_DL\\Phase3_GitHub\\Dataset\\DEM_SAMPLE.npy")
sar = np.load("D:\\M.Tech\\Landslide_DL\\Phase3_GitHub\\Dataset\\SAR_SAMPLE.npy")
mask = np.load("D:\\M.Tech\\Landslide_DL\\Phase3_GitHub\\Dataset\\MASK_SAMPLE.npy")
rgbi = np.load("D:\\M.Tech\\Landslide_DL\\Phase3_GitHub\\Dataset\\RGBI_SAMPLE.npy")

datasets = [mss, rgbi, mask, dem, sar]

dataset_names = ['Multispectral Layers', 'RGBI', 'Mask Layers', 'DEM Layers', 'SAR Layers']

for i in range(len(datasets)):
    print(f'{dataset_names[i]}: {datasets[i].shape}')    

In [ ]:
### PLOT SOME SAMPLES ###

# Input Titles
titles = [
    'Landslide Mask',
    'BI - Post', 'NDVI - Post', 'NDWI - Post', 'NIR - Post',
    'BI - Pre', 'NDVI - Pre', 'NDWI - Pre', 'NIR - Pre',
    'VV - Post', 'VH - Post', 'VV-VH Diff - Post',
    'VV - Pre', 'VH - Pre', 'VV-VH Diff - Pre',
    'Elevation', 'Slope', 'Aspect-Sin', 'Aspect-Cos', 'TWI'
]

# Order in which DEM layers need to be plotted 
dem_order_map = {
    'Aspect-Sin': 0,
    'Aspect-Cos': 1,
    'Elevation': 2,
    'Slope': 3,
    'TWI': 4
}

# Function to get the respective layer for each index
def get_image(i, idx):

    if idx == 0: return mask[i]    # Mask

    # Post-event optical inputs
    elif 1 <= idx <= 4:
        post_map = [1, 2, 3, 0]  # BI, NDVI, NDWI, NIR
        return mss[i, :, :, post_map[idx - 1]]

    # Pre-event optical inputs
    elif 5 <= idx <= 8:
        pre_map = [5, 6, 7, 4]  # BI, NDVI, NDWI, NIR
        return mss[i, :, :, pre_map[idx - 5]]

    # Post-event SAR inputs
    elif 9 <= idx <= 11: return sar[i, :, :, idx - 9]

    # Pre-event SAR inputs
    elif 12 <= idx <= 14: return sar[i, :, :, idx - 9]

    # DEM inputs
    elif 15 <= idx <= 19:
        dem_idx = dem_order_map[titles[idx]]
        return dem[i, :, :, dem_idx]

    else: return None

# Function for RGB stretch (-1 to 1 with -9 as Nan)
def stretch_rgb(rgb):
    rgb = rgb.astype(np.float32)
    out = np.zeros_like(rgb)

    for b in range(3):
        band = rgb[:, :, b]
        valid = band[band != -9]
        if valid.size > 0:
            mn, mx = valid.min(), valid.max()
            out[:, :, b] = (band - mn) / (mx - mn + 1e-6)
    return out

# Loop through samples and plot
n = 10     # No. of samples to plot

for i in range(n):

    fig = plt.figure(figsize=(7, 12))

    # Row 1: Post-event RGB, pre-event RGB, landslide mask
    plt.subplot(6, 4, 1)
    plt.imshow(stretch_rgb(rgbi[i, :, :, :3]))
    plt.title('RGB - Post', fontsize=12)
    plt.axis('off')

    plt.subplot(6, 4, 2)
    plt.imshow(stretch_rgb(rgbi[i, :, :, 4:7]))
    plt.title('RGB - Pre', fontsize=12)
    plt.axis('off')

    plt.subplot(6, 4, 3)
    plt.imshow(np.ma.masked_where(mask[i] == -9, mask[i]), cmap='gray')
    plt.title('Landslide Mask', fontsize=12)
    plt.axis('off')

    # Model inputs
    plot_pos = 5
    bi_im = None

    for idx in range(1, 20):

        ax = plt.subplot(6, 4, plot_pos)
        img = get_image(i, idx)
        img_masked = np.ma.masked_where(img == -9, img)

        im = ax.imshow(img_masked)
        ax.set_title(titles[idx], fontsize=12)
        ax.axis('off')

        if idx == 1:
            bi_im = im

        plot_pos += 1

    # Plot the colorbar in the bottom
    if bi_im is not None:
        cax = fig.add_axes([0.15, 0.03, 0.7, 0.02])  
        # [left, bottom, width, height]

        cbar = fig.colorbar(bi_im, cax=cax, orientation='horizontal')

    plt.tight_layout(rect=[0, 0.06, 1, 1])  # leave space for colorbar
    plt.show()

In [ ]:
# Model weight paths
model_1_weights = "D:\\M.Tech\\Landslide_DL\\Phase3_GitHub\\Weights\\unet_filters_16_batch_4_lr_0.0001_epoch_200.weights.h5"
model_2_weights = "D:\\M.Tech\\Landslide_DL\\Phase3_GitHub\\Weights\\unetpp_filters_32_batch_8_lr_0.0005_epoch_100.weights.h5"
model_3_weights = "D:\\M.Tech\\Landslide_DL\\Phase3_GitHub\\Weights\\unetppeca_filters_32_batch_32_lr_0.0005_epoch_100.weights.h5"
model_4_weights = "D:\\M.Tech\\Landslide_DL\\Phase3_GitHub\\Weights\\unetppdeepsupeca_filters_8_batch_4_lr_0.0005_epoch_110.weights.h5"
model_5_weights = "D:\\M.Tech\\Landslide_DL\\Phase3_GitHub\\Weights\\unetppdeepsupecaaspp_filters_32_batch_4_lr_0.0001_epoch_200.weights.h5"

# Prepare the models and parameters for looping over them
models = [model_1, model_2, model_3, model_4, model_5]
names = ['Triple U-Net', 'Triple U-Net++', 'Triple U-Net++ w/ ECA', 'Triple U-Net++ w/ ECA, Deep Supervision', 'Triple U-Net++ w/ ECA, Deep Supervision, ASPP']
weights = [model_1_weights, model_2_weights, model_3_weights, model_4_weights, model_5_weights]
filters = [16, 32, 32, 8, 32]
learning_rates = [0.0001, 0.0005, 0.0005, 0.0005, 0.0001]
preds = []

In [ ]:
# Loop over 5 models
for i in range(5):
    # Load the model with weights
    model = models[i]
    model = model((128, 128, 8), (32, 32, 6), (32, 32, 5), filters[i], learning_rates[i], loss = tversky_loss)
    model.load_weights(weights[i])

    # Get predictions
    pred = model.predict({"input_img": mss, "input_dem": dem, 'input_sar': sar})

    if isinstance(pred, (list, tuple)): pred_final = (pred[0] > 0.5).astype(int)
    else: pred_final = (pred > 0.5).astype(int)

    preds.append(pred_final)

    # Compute and print metrics
    precision = precision_score(mask.flatten(), pred_final.flatten())
    recall = recall_score(mask.flatten(), pred_final.flatten())
    f1 = f1_score(mask.flatten(), pred_final.flatten())
    iou = jaccard_score(mask.flatten(), pred_final.flatten())
    mcc = matthews_corrcoef(mask.flatten(), pred_final.flatten())

    print(names[i])
    print(f'Precision: {precision*100:.4f} %')
    print(f'Recall: {recall*100:.4f} %')
    print(f'F1: {f1*100:.4f} %')
    print(f'IoU: {iou*100:.4f} %')
    print(f'MCC: {mcc*100:.4f} %')

In [ ]:
# Plot the results
n = mss.shape[0]
col_titles = ['RGB Image', 'Landslide Mask'] + [f'Model {i+1}' for i in range(len(names))]
fig, axes = plt.subplots(n, 7, figsize=(12, 1.8*n))

for row, i in enumerate(range(n)):

    # RGB
    axes[row, 0].imshow(stretch_rgb(rgbi[i, :, :, :3]))
    axes[row, 0].axis('off')

    # Mask
    axes[row, 1].imshow(mask[i])
    axes[row, 1].axis('off')

    # Model predictions
    for col, _ in enumerate(names):
        axes[row, 2 + col].imshow(preds[col][i])
        axes[row, 2 + col].axis('off')

    # Titles only for first row
    if row == 0:
        for c in range(7):
            axes[row, c].set_title(col_titles[c], fontsize=14)

plt.tight_layout()
plt.show()